In [55]:
# read data files 
import pandas as pd 
df_TUE = pd.read_csv('/mimer/NOBACKUP/groups/naiss2024-22-903/WASP_NLP/project/data/TUE/all_CS_courses.csv')
#df_test = pd.read_csv('/mimer/NOBACKUP/groups/naiss2024-22-903/WASP_NLP/project/data/test_courses.csv')
#print(df_test['course_code'])
df_OVL = pd.read_csv('/mimer/NOBACKUP/groups/naiss2024-22-903/WASP_NLP/project/data/TUE/all_overlap_courses.csv')
 
filename = "/mimer/NOBACKUP/groups/naiss2024-22-903/WASP_NLP/project/data/TUE/overlap_table.csv"
df_overlap = pd.read_csv(filename)
print(len(df_overlap))

print(len(df_TUE))
print(len(df_OVL))
print(df_TUE.head())

74
74
91
                                             title credits             level  \
0                   Measure and Probability Theory    5 EC          Advanced   
1                    Automata and Formal Languages    5 EC          Bachelor   
2                  Linear Algebra for Data Science    5 EC               BSc   
3          Impact of technology engineering ethics    5 EC               NaN   
4  Stochastic Optimization Models for Data Science    5 EC  Bachelor College   

                                         description  \
0  In this course, you will discover how measure ...   
1  This course covers fundamental concepts and th...   
2  Systems of linear equations are often used in ...   
3  Ethics of Technology and Engineering introduce...   
4  All information regarding this course (incl. l...   

                                              topics  \
0  ['Measures, measurable functions and sigma-alg...   
1  ['deterministic finite automaton (DFA)', 'non-...   
2  ['

In [154]:
print(df_TUE['description'].loc[0])
print(df_TUE['learning_outcomes'].loc[0])
print(df_TUE['topics'].loc[0])

In this course, you will discover how measure theory brings together key concepts in analysis and probability theory. We will demonstrate why this unified language is essential, teach you the necessary tools, and illustrate how measure theory connects analysis and probability theory.
['Define the Lebesgue integral and explain its differences from the Riemann integral', 'Compute Lebesgue integral of possibly non-Riemann integrable functions', 'Define sigma-algebras and explain their significance in measure theory and their use in probability theory', 'Define measurable functions and determine if a given function is measurable', 'State and apply key convergence theorems such as Fatou’s, monotone and dominated convergence', 'State and apply Fubini-Tonelli theorems', 'Describe and differentiate between the various notions of function convergence, and prove that a sequence of functions converges according to one of these notions', 'Define conditional probability/expectation in measure-theor

In [24]:
# Add a column showing which courses are duplicated
#df_TUE['duplicate_course_codes'] = df_TUE['course_code'].apply(
#    lambda x: x if x in df2['course_code'].values else None
#)

# Or get a boolean mask and the actual values
df_TUE['is_duplicate'] = df_TUE['course_code'].isin(df_OVL['course_code'])
df_TUE['matching_code'] = df_TUE['course_code'].where(df_TUE['is_duplicate'])

# Count duplicates
duplicate_count = df_TUE['is_duplicate'].sum()
print(f"Found {duplicate_count} duplicates out of {len(df_TUE)} courses")
print(df_TUE['matching_code'])

Found 29 duplicates out of 74 courses
0        NaN
1        NaN
2     JBM075
3     0LVX10
4        NaN
       ...  
69       NaN
70       NaN
71    2IRR50
72       NaN
73    2MBS10
Name: matching_code, Length: 74, dtype: object


In [4]:
# Load sBERT tokenizer 
import nltk 
nltk.download('punkt') 
nltk.download('punkt_tab')
from nltk.tokenize import sent_tokenize 
from sentence_transformers import SentenceTransformer 
import numpy as np 
import torch 

model_name = "sentence-transformers/all-mpnet-base-v2"   # sBERT model
model = SentenceTransformer(model_name)
tokenizer = model.tokenizer
max_tokens = 256  # model limit

import itertools
import re
from typing import List, Dict, Set
import pandas as pd 

def normalize(title: str) -> str:
    """Normalize text"""
    if pd.isna(title):
        return ""
    
    title = str(title)
    title = title.lower() # Convert to lowercase
    title = title.replace('&', 'and') # Replace '&' with 'and'
    title = re.sub(r'[^\w\s-]', ' ', title) # Remove special characters but keep spaces and hyphens
    title = re.sub(r'\b(and|or|the|of|for|in|to|a|an|with|at|by)\b', ' ', title) # Remove common conjunctions as standalone words
    title = re.sub(r'\s+', ' ', title).strip() # Clean up extra spaces
    title = re.sub(r's\b', '', title) # Remove trailing 's' for plurals 
    return title

[nltk_data] Downloading package punkt to
[nltk_data]     /cephyr/users/nijdam/Alvis/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /cephyr/users/nijdam/Alvis/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
/mimer/NOBACKUP/groups/naiss2024-22-903/anaconda3/envs/NLP/lib/python3.8/site-packages/transformers/utils/hub.py:128: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [2]:
import re 
def embed_paragraphs(model, tokenizer, paragraphs):
    if len(paragraphs) == 0:
        return np.zeros(model.get_sentence_embedding_dimension())
    
    embeddings = model.encode(paragraphs, convert_to_numpy=True, normalize_embeddings=True)
    return np.mean(embeddings, axis=0)

def embed_paragraphs2(model, tokenizer, paragraphs):
    if len(paragraphs) == 0:
        return np.zeros(model.get_sentence_embedding_dimension())
    else: 
        paragraphs = [normalize(paragraph) for paragraph in paragraphs]
    
    embeddings = model.encode(paragraphs, convert_to_numpy=True, normalize_embeddings=False)
    return np.mean(embeddings, axis=0)

def split_sentences(text): 
    sentences = re.split(r'\.\s+(?=[A-Z])', text)
    
    # Re-add the period to each sentence (except maybe last if you want)
    sentences = [s.strip() + '.' for s in sentences if s]
    
    #for s in sentences:
      #  print(s)
    return sentences 

In [137]:
# also extract LO and topic embs 
# now extract the embeddings for all courses 
course_embs = [] 
course_names = [] 
for i,course in enumerate(df_TUE['description']): #[0:23]: 
    if isinstance(course,str):            
        out2 = embed_paragraphs2(model,tokenizer, split_sentences(course)) 
        course_embs.append(out2) 
        course_names.append(df_TUE['course_code'][i])

# to do: fix this! one-one pairing of LOs and topics is necessary. 
LO_embs = [] 
course_names2 = [] 
for i,course in enumerate(df_TUE['learning_outcomes']): #[0:23]: 
    if isinstance(course,str):         
        split = split_sentences(course)
        out2 = embed_paragraphs(model,tokenizer,split ) 
        LO_embs.append(out2) 
        course_names2.append(df_TUE['course_code'][i])

topic_embs = [] 
course_names3 = [] 
for i,course in enumerate(df_TUE['topics']): #[0:23]: 
    if isinstance(course,str):         
        split = split_sentences(course)
        out2 = embed_paragraphs(model,tokenizer,split ) 
        topic_embs.append(out2) 
        course_names3.append(df_TUE['course_code'][i])

title_embs = [] 
course_names4 = [] 
for i,course in enumerate(df_TUE['title']): #[0:23]: 
    if isinstance(course,str):         
        #split = split_sentences(course)
        out2 = model.encode(course, convert_to_numpy=True, normalize_embeddings=True)
        title_embs.append(out2) 
        course_names4.append(df_TUE['course_code'][i])

In [156]:
print(LO_embs[0].shape)

(768,)


In [138]:
# do the same for df_OVL
# also extract LO and topic embs 
# now extract the embeddings for all courses 
course_embsb = [] 
course_namesb = [] 
for i,course in enumerate(df_OVL['description']): #[0:23]: 
    if isinstance(course,str):            
        out2 = embed_paragraphs2(model,tokenizer, split_sentences(course)) 
        course_embsb.append(out2) 
        course_namesb.append(df_OVL['course_code'][i])

LO_embsb = [] 
course_names2b = [] 
for i,course in enumerate(df_OVL['learning_outcomes']): #[0:23]: 
    if isinstance(course,str):         
        split = split_sentences(course)
        out2 = embed_paragraphs(model,tokenizer,split ) 
        LO_embsb.append(out2) 
        course_names2b.append(df_OVL['course_code'][i])

topic_embsb = [] 
course_names3b = [] 
for i,course in enumerate(df_OVL['topics']): #[0:23]: 
    if isinstance(course,str):         
        split = split_sentences(course)
        out2 = embed_paragraphs(model,tokenizer,split ) 
        topic_embsb.append(out2) 
        course_names3b.append(df_OVL['course_code'][i])

title_embsb = [] 
course_names4b = [] 
for i,course in enumerate(df_OVL['title']): #[0:23]: 
    if isinstance(course,str):         
        #split = split_sentences(course)
        out2 = model.encode(course, convert_to_numpy=True, normalize_embeddings=True)
        title_embsb.append(out2) 
        course_names4b.append(df_OVL['course_code'][i])

In [139]:
from sklearn.metrics.pairwise import cosine_similarity
embs = np.array(course_embs) 
embs_ov = np.array(course_embsb)
print(np.shape(embs))
print(np.shape(embs_ov))
print(len(course_names))
print(len(course_namesb))

(70, 768)
(57, 768)
70
57


In [140]:
cos_sim = cosine_similarity(embs, embs_ov)
print(np.shape(cos_sim))

(70, 57)


In [141]:
# create mask 
mask = np.equal.outer(course_names, course_namesb)
print(sum(sum(mask)))
# mask out same courses 
    
# Apply mask if provided
#if mask is not None:
# Mask out values where mask is False
cos_sim_masked = np.where(~mask, cos_sim, 0) # set all overlapping values to 0 so they don't get selected
chosen = np.argmax(cos_sim_masked, axis=1)
score = np.max(cos_sim_masked, axis=1)
print(len(chosen))
print(len(score))
th = 0.75 
print(score)

29
70
70
[0.6885663  0.53914225 0.6232754  0.90099394 0.9999998  0.61297
 0.69159913 0.81359696 0.6425667  0.67827827 0.5892164  1.0000001
 0.7525666  0.9259472  0.566776   0.4023767  0.69390804 0.7470361
 0.46916106 1.0000001  0.6902325  0.6541771  0.7588364  0.5494664
 0.5510121  0.98828006 0.6356896  0.6779479  0.6716978  0.7179041
 0.660623   0.7470361  0.7737952  0.7151469  0.8882859  0.94948363
 0.56028676 0.6357829  1.         0.65616703 0.7525666  0.8504908
 0.63657    0.8385446  0.6695766  0.6253547  0.61555594 0.85284853
 0.7616045  0.99999994 0.71856153 0.6962543  0.9882801  0.5704726
 0.5285184  0.6510426  0.77948844 0.4401577  0.6984723  0.6414118
 0.758877   0.70566094 0.8938199  0.591845   0.6824397  0.74113953
 0.8641968  0.71439505 0.6651041  0.8197117 ]


In [21]:
chosen = np.argmax(cos_sim, axis=1)
score = np.max(cos_sim, axis=1)
print(len(chosen))
print(len(score))
th = 0.4 
print(score)
print([i for i in score if i <=0.95])

70
70
[0.6885663  0.53914225 1.         0.9999999  0.9999998  0.61297
 0.9999999  0.98539746 1.0000002  0.67827827 0.5892164  1.0000001
 0.9999999  0.99999994 0.566776   0.4023767  0.69390804 0.7470361
 0.46916106 1.0000001  0.6902325  1.0000002  0.7588364  0.5494664
 0.5510121  1.0000002  0.9999999  0.6779479  0.6716978  0.7179041
 1.         0.7470361  0.7737952  0.7151469  1.0000002  0.94948363
 1.0000001  0.9999999  1.         0.65616703 1.0000001  0.9896882
 0.63657    0.8385446  0.6695766  0.96085835 1.0000002  0.85284853
 1.0000001  0.99999994 1.0000001  0.6962543  0.99999976 0.5704726
 0.5285184  0.6510426  0.94650537 0.4401577  0.6984723  1.
 0.758877   0.70566094 0.8938199  0.591845   0.93917996 0.74113953
 0.8641968  0.71439505 1.0000002  1.0000001 ]
[0.6885663, 0.53914225, 0.61297, 0.67827827, 0.5892164, 0.566776, 0.4023767, 0.69390804, 0.7470361, 0.46916106, 0.6902325, 0.7588364, 0.5494664, 0.5510121, 0.6779479, 0.6716978, 0.7179041, 0.7470361, 0.7737952, 0.7151469, 0.9494

In [54]:
print(course_names[3])
print(course_namesb[chosen[3]])
th = 0.7
print('value: ',[x for x in cos_sim_masked[3,:] if x >= th])
val = [x for x in cos_sim_masked[3,:] if x >= th]
print('code: ',[course_namesb[i] for i, x in enumerate(cos_sim_masked[3,:]) if x >= th])
chosen_code = [course_namesb[i] for i, x in enumerate(cos_sim_masked[3,:]) if x >= th]

0LVX10
0LVX30
value:  [0.90099394, 0.7402753]
code:  ['0LVX30', '0LVX20']


In [57]:
# manually input all old codes 
old_codes = ['2WN30', '2WF50', 'JBI040', '2WA30', 'JBM060', '2WF70', '2IT90', '2ILH0', 
             '2WA80', 'JBM020', 'JBM010', 'JBI025', 'JBI020', 'JBM090', 'JBM090-B-5', '2IAB0', '2WS30', '2WF60', 
             '2WF80', '2WN20', 'JBL120', '2WF20', '2WF20', '2WF30', '2DBI00', '2DRR00', 'JBM070', '2WO020', 
             '2WH60', '2WAG0', '2IC60', '2WN40', '2WA90', '2WS20', '2WH20', '2WB40', '2WS40', '2IC60', '2WF40', 
             '2IIG0', '2WB20', '2WB50', '2WA70', '2WA60'] 
print(len(old_codes))

44


In [142]:
# compute TP/FP/TN/FN 
result = df_overlap.loc[df_overlap['New_code'] == course_names[3], 'Overlap'] 
print(result)
codes = str(result.iloc[0]).split()
true_overlap = [code for code in codes if code not in old_codes and code != 'nan']
print(true_overlap)

36    0SAB0 JBG000 0LVX30
Name: Overlap, dtype: object
['0SAB0', 'JBG000', '0LVX30']


In [87]:
from sklearn.metrics.pairwise import cosine_similarity
def compute_cos_sim(course_embs, course_embsb, course_names, course_namesb): 
    embs = np.array(course_embs) 
    embs_ov = np.array(course_embsb)
    # compute cosine similarity 
    cos_sim = cosine_similarity(embs, embs_ov)
    # create mask 
    mask = np.equal.outer(course_names, course_namesb)
    # mask out same courses 
    cos_sim_masked = np.where(~mask, cos_sim, 0) # set all overlapping values to 0 so they don't get selected
    return cos_sim_masked 

cos_sim_masked = compute_cos_sim(course_embs, course_embsb, course_names, course_namesb)

In [143]:
# Or if you want to use all ground truths, aggregate them

def compute_metrics(th, course_names, course_namesb, cos_sim_masked, df_overlap):
    all_TP = 0
    all_FP = 0
    all_FN = 0
    all_TN = 0
    
    for i in range(len(course_names)): 
        #print('value: ',[x for x in cos_sim_masked[i,:] if x >= th])
        val = [x for x in cos_sim_masked[i,:] if x >= th]
        #print('code: ',[course_namesb[i] for i, x in enumerate(cos_sim_masked[i,:]) if x >= th])
        chosen_code = [course_namesb[i] for i, x in enumerate(cos_sim_masked[i,:]) if x >= th]
    
        result = df_overlap.loc[df_overlap['New_code'] == course_names[i], 'Overlap'] 
        #print(result)
        codes = str(result.iloc[0]).split()
        true_overlap = [code for code in codes if code not in old_codes and code != 'nan']
        #print('label: ',true_overlap)
        
        all_codes = set(course_namesb)
        pred_set = set(chosen_code) 
        gt_set = set(true_overlap) 
        
        # Calculate metrics
        TP = len(gt_set & pred_set)  # Intersection: {'0LVX30'} -> 1
        FP = len(pred_set - gt_set)  # In predicted but not ground truth: {'0LVX20'} -> 1
        FN = len(gt_set - pred_set)  # In ground truth but not predicted: {'0SAB0', 'JBG000'} -> 2
        TN = len(all_codes - (gt_set | pred_set))  # In neither: nothing -> 0
        
        all_TP += TP
        all_FP += FP
        all_FN += FN
        all_TN += TN
        #print('TP: ',TP)
        #print('FP: ',FP)
        #print('FN: ',FN)
        #print('TN: ',TN)
        #break 
    precision = all_TP / (all_TP + all_FP) if (all_TP + all_FP) > 0 else 0
    recall = all_TP / (all_TP + all_FN) if (all_TP + all_FN) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    return precision, recall, f1

def find_best_th_GS(course_names, course_namesb, cos_sim_masked, df_overlap): 
    best_f1 = 0 
    best_th = 0 
    for th in np.arange(0.05, 1, 0.05): 
        precision, recall, f1 = compute_metrics(th, course_names, course_namesb, cos_sim_masked, df_overlap)
        if f1 >= best_f1: 
            best_f1 = f1 
            best_th = th 
    print('best_f1: ',best_f1)
    print('best_th: ',best_th)

In [144]:
print('description only: ')
cos_sim_masked_desc = compute_cos_sim(course_embs, course_embsb, course_names, course_namesb)
find_best_th_GS(course_names, course_namesb, cos_sim_masked_desc, df_overlap)

print('LO only: ')
cos_sim_masked_lo = compute_cos_sim(LO_embs, LO_embsb, course_names2, course_names2b)
find_best_th_GS(course_names2, course_names2b, cos_sim_masked_lo, df_overlap)

print('topic only: ')
cos_sim_masked_topic = compute_cos_sim(topic_embs, topic_embsb, course_names3, course_names3b)
find_best_th_GS(course_names3, course_names3b, cos_sim_masked_topic, df_overlap)

print('title only: ')
cos_sim_masked_title = compute_cos_sim(title_embs, title_embsb, course_names4, course_names4b)
find_best_th_GS(course_names4, course_names4b, cos_sim_masked_title, df_overlap)

description only: 
best_f1:  0.27007299270072993
best_th:  0.6500000000000001
LO only: 
best_f1:  0.10819672131147541
best_th:  0.7000000000000001
topic only: 
best_f1:  0.14862681744749595
best_th:  0.7000000000000001
title only: 
best_f1:  0.3189655172413793
best_th:  0.55


so from the perspective of the course description, we should set a threshold of 0.65 

In [149]:
# Get all course names
all_names = np.unique(np.concatenate([course_names, course_names2, course_names3, course_names4]))
all_names_b = np.unique(np.concatenate([course_namesb, course_names2b, course_names3b, course_names4b]))

# Initialize
combined = np.zeros((len(all_names), len(all_names_b)))
total_w = np.zeros((len(all_names), len(all_names_b)))

# Helper function to map names to indices
def fill_similarity(sim, names, names_b, weight):
    row_map = {name: i for i, name in enumerate(names)}
    col_map = {name: i for i, name in enumerate(names_b)}
    
    for i, name in enumerate(all_names):
        if name in row_map:
            for j, name_b in enumerate(all_names_b):
                if name_b in col_map:
                    combined[i, j] += weight * sim[row_map[name], col_map[name_b]]
                    total_w[i, j] += weight

# Fill each field
th= 0.6 #optimal
weights = [0.7, 0, 0, 0.3] #optimal
fill_similarity(cos_sim_masked_desc, course_names, course_namesb, weights[0])
fill_similarity(cos_sim_masked_lo, course_names2, course_names2b, weights[1])
fill_similarity(cos_sim_masked_topic, course_names3, course_names3b,weights[2])
fill_similarity(cos_sim_masked_title, course_names4, course_names4b, weights[3])

# Normalize
total_w = np.maximum(total_w, 1e-10)
combined_sim = combined / total_w

# Use it
find_best_th_GS(all_names, all_names_b, combined_sim, df_overlap)
for i in range(len(course_names)): 
    print('course name: ',course_names[i])
    print('value: ',[x for x in cos_sim_masked[i,:] if x >= th])
    val = [x for x in cos_sim_masked[i,:] if x >= th]
    print('code: ',[course_namesb[i] for i, x in enumerate(cos_sim_masked[i,:]) if x >= th])
    chosen_code = [course_namesb[i] for i, x in enumerate(cos_sim_masked[i,:]) if x >= th]

    result = df_overlap.loc[df_overlap['New_code'] == course_names[i], 'Overlap'] 
    #print(result.iloc[0])
    codes = str(result.iloc[0]).split()
    true_overlap = [code for code in codes if code not in old_codes and code != 'nan']
    print('label: ',true_overlap)
    
    all_codes = set(course_namesb)
    pred_set = set(chosen_code) 
    gt_set = set(true_overlap) 
    
    # Calculate metrics
    TP = len(gt_set & pred_set)  # Intersection: {'0LVX30'} -> 1
    FP = len(pred_set - gt_set)  # In predicted but not ground truth: {'0LVX20'} -> 1
    FN = len(gt_set - pred_set)  # In ground truth but not predicted: {'0SAB0', 'JBG000'} -> 2
    TN = len(all_codes - (gt_set | pred_set))  # In neither: nothing -> 0
    


best_f1:  0.37004405286343617
best_th:  0.6000000000000001
course name:  2MBA70
value:  [0.6000817, 0.66269547, 0.61352944, 0.6803509, 0.6804261, 0.6885663]
code:  ['2DRR10', '2DD80', '2MBS20', '2MBS10', '2DI90', '2DD40']
label:  []
course name:  2IRR90
value:  []
code:  []
label:  ['2ITS90']
course name:  JBM075
value:  [0.6232754]
code:  ['2MBA20']
label:  []
course name:  0LVX10
value:  [0.90099394, 0.7402753, 0.65593493, 0.6607655]
code:  ['0LVX30', '0LVX20', '0HV150', 'JBG000']
label:  ['0SAB0', 'JBG000', '0LVX30']
course name:  JBM100
value:  [0.9583173, 0.9999998, 0.8120312]
code:  ['JBM090;', 'JBM080', 'JBM035']
label:  ['2MBS30']
course name:  2MBA40
value:  [0.61297, 0.6053315]
code:  ['2WAB0', '2WBB0']
label:  ['JBM080']
course name:  JBG000
value:  [0.6607655, 0.69159913, 0.669663, 0.66430616, 0.62857926]
code:  ['0LVX10', '0LVX30', 'JBI050', '0LVX20', 'JBI030']
label:  ['0SAB0', '0LVX10', '0LVX20', '0LVX30', '0LVX40']
course name:  2MBS20
value:  [0.6245693, 0.70478225, 0.

In [126]:
from itertools import product
def grid_search_weights(sim_dict, name_pairs, all_names, all_names_b, df_overlap, step=0.25):
    """
    Grid search over weights with given step size
    """
    weight_values = np.arange(0, 1.01, step)
    best_f1_best = -1
    best_weights = None
    best_th_best = None
    best_combined = None
    
    for w_desc, w_lo, w_topic, w_title in product(weight_values, repeat=4):
        total = w_desc + w_lo + w_topic + w_title
        if abs(total - 1.0) > step/2:  # Must sum to ~1.0
            continue
        if total == 0:
            continue
        
        # Compute weighted combination
        combined = np.zeros((len(all_names), len(all_names_b)))
        total_w = np.zeros((len(all_names), len(all_names_b)))
        
        weights = {'desc': w_desc, 'lo': w_lo, 'topic': w_topic, 'title': w_title}
        
        for field, weight in weights.items():
            if weight == 0:
                continue
                
            sim = sim_dict[field]
            names, names_b = name_pairs[field]
            
            row_map = {name: i for i, name in enumerate(names)}
            col_map = {name: i for i, name in enumerate(names_b)}
            
            for i, name in enumerate(all_names):
                if name in row_map:
                    for j, name_b in enumerate(all_names_b):
                        if name_b in col_map:
                            combined[i, j] += weight * sim[row_map[name], col_map[name_b]]
                            total_w[i, j] += weight
        
        total_w = np.maximum(total_w, 1e-10)
        combined_sim = combined / total_w
        
        # Evaluate
        best_f1, best_t = find_best_th_GS(all_names, all_names_b, combined_sim, df_overlap)
        
        if best_f1 > best_f1_best:
            #print('yes: ',best_f1)
            best_f1_best = best_f1
            best_weights = (w_desc, w_lo, w_topic, w_title)
            best_th = best_t
            best_combined = combined_sim
    
    return best_weights, best_f1_best, best_th, best_combined

def find_best_th_GS(course_names, course_namesb, cos_sim_masked, df_overlap): 
    best_f1 = 0 
    best_th = 0 
    for th in np.arange(0.05, 1, 0.05): 
        precision, recall, f1 = compute_metrics(th, course_names, course_namesb, cos_sim_masked, df_overlap)
        if f1 >= best_f1: 
            best_f1 = f1 
            best_th = th 
   # print('best_f1: ',best_f1)
    return best_f1, best_th 
    #print('best_th: ',best_th)

# Run grid search
best_weights, best_f1, best_th, best_combined = grid_search_weights(
    sim_dict, name_pairs, all_names, all_names_b, df_overlap, step=0.25
)

print(f"Best weights: desc={best_weights[0]:.2f}, lo={best_weights[1]:.2f}, topic={best_weights[2]:.2f}, title={best_weights[3]:.2f}")
print(f"Best F1: {best_f1:.4f} at threshold {best_th:.4f}")

Best weights: desc=0.25, lo=0.00, topic=0.00, title=0.75
Best F1: 0.3652 at threshold 0.5500


In [127]:
# Run grid search (more granular) 
best_weights, best_f1, best_th, best_combined = grid_search_weights(
    sim_dict, name_pairs, all_names, all_names_b, df_overlap, step=0.05
)

print(f"Best weights: desc={best_weights[0]:.2f}, lo={best_weights[1]:.2f}, topic={best_weights[2]:.2f}, title={best_weights[3]:.2f}")
print(f"Best F1: {best_f1:.4f} at threshold {best_th:.4f}")

Best weights: desc=0.70, lo=0.00, topic=0.00, title=0.30
Best F1: 0.3700 at threshold 0.6000


In [ ]:
# inspect results 


In [ ]:
# precision instead 

In [134]:
from itertools import product
def grid_search_weights(sim_dict, name_pairs, all_names, all_names_b, df_overlap, step=0.25):
    """
    Grid search over weights with given step size
    """
    weight_values = np.arange(0, 1.01, step)
    best_f1_best = -1
    best_weights = None
    best_th_best = None
    best_combined = None
    
    for w_desc, w_lo, w_topic, w_title in product(weight_values, repeat=4):
        total = w_desc + w_lo + w_topic + w_title
        if abs(total - 1.0) > step/2:  # Must sum to ~1.0
            continue
        if total == 0:
            continue
        
        # Compute weighted combination
        combined = np.zeros((len(all_names), len(all_names_b)))
        total_w = np.zeros((len(all_names), len(all_names_b)))
        
        weights = {'desc': w_desc, 'lo': w_lo, 'topic': w_topic, 'title': w_title}
        
        for field, weight in weights.items():
            if weight == 0:
                continue
                
            sim = sim_dict[field]
            names, names_b = name_pairs[field]
            
            row_map = {name: i for i, name in enumerate(names)}
            col_map = {name: i for i, name in enumerate(names_b)}
            
            for i, name in enumerate(all_names):
                if name in row_map:
                    for j, name_b in enumerate(all_names_b):
                        if name_b in col_map:
                            combined[i, j] += weight * sim[row_map[name], col_map[name_b]]
                            total_w[i, j] += weight
        
        total_w = np.maximum(total_w, 1e-10)
        combined_sim = combined / total_w
        
        # Evaluate
        best_f1, best_t = find_best_th_GS(all_names, all_names_b, combined_sim, df_overlap)
        
        if best_f1 > best_f1_best:
            #print('yes: ',best_f1)
            best_f1_best = best_f1
            best_weights = (w_desc, w_lo, w_topic, w_title)
            best_th = best_t
            best_combined = combined_sim
    
    return best_weights, best_f1_best, best_th, best_combined

def find_best_th_GS(course_names, course_namesb, cos_sim_masked, df_overlap): 
    best_f1 = 0 
    best_th = 0 
    for th in np.arange(0.05, 1, 0.05): 
        precision, recall, f1 = compute_metrics(th, course_names, course_namesb, cos_sim_masked, df_overlap)
        if recall >= best_f1: 
            best_f1 = recall 
            best_th = th 
    print('best_f1: ',best_f1)
    return best_f1, best_th 
    #print('best_th: ',best_th)

# Run grid search
best_weights, best_f1, best_th, best_combined = grid_search_weights(
    sim_dict, name_pairs, all_names, all_names_b, df_overlap, step=0.25
)

print(f"Best weights: desc={best_weights[0]:.2f}, lo={best_weights[1]:.2f}, topic={best_weights[2]:.2f}, title={best_weights[3]:.2f}")
print(f"Best F1: {best_f1:.4f} at threshold {best_th:.4f}")

best_f1:  0.6578947368421053
best_f1:  0.9298245614035088
best_f1:  0.9298245614035088
best_f1:  0.9298245614035088
best_f1:  0.9035087719298246
best_f1:  0.8859649122807017
best_f1:  0.9122807017543859
best_f1:  0.9210526315789473
best_f1:  0.8947368421052632
best_f1:  0.8859649122807017
best_f1:  0.9122807017543859
best_f1:  0.8859649122807017
best_f1:  0.868421052631579
best_f1:  0.8771929824561403
best_f1:  0.7807017543859649
best_f1:  0.6578947368421053
best_f1:  0.9298245614035088
best_f1:  0.9298245614035088
best_f1:  0.9298245614035088
best_f1:  0.8859649122807017
best_f1:  0.9122807017543859
best_f1:  0.9122807017543859
best_f1:  0.8859649122807017
best_f1:  0.8947368421052632
best_f1:  0.8596491228070176
best_f1:  0.6578947368421053
best_f1:  0.9298245614035088
best_f1:  0.9298245614035088
best_f1:  0.8859649122807017
best_f1:  0.9035087719298246
best_f1:  0.8771929824561403
best_f1:  0.6578947368421053
best_f1:  0.9298245614035088
best_f1:  0.8771929824561403
best_f1:  0.640